# TwinAero: NASA C-MAPSS FD004 Analysis & Digital Twin Evaluation
**Dataset:** NASA Commercial Modular Aero-Propulsion System Simulation (C-MAPSS) - FD004  
**Characteristics:** 6 Operating Flight Regimes, Dual Degradation Modes (HPC & Fan), 248 Train / 249 Test Engines


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
sys.path.append("..")
from ml.preprocessing.loader import CMAPSSDataLoader
from ml.rul.predictor import RULPredictor
from ml.uncertainty.conformal import ConformalPredictor


## 1. Data Ingestion & 6 Operating Flight Regimes
FD004 varies across Altitude, Mach number, and Throttle Resolver Angle (TRA).

In [ ]:
loader = CMAPSSDataLoader(data_dir="../data/raw")
train_df, test_df, rul_true = loader.load_dataset("FD004")
print(f"Training instances: {len(train_df):,}")
print(f"Test instances:     {len(test_df):,}")
print("Flight Regimes distribution:
", train_df["regime"].value_counts().sort_index())

## 2. Regime-Wise Standardization
Eliminating flight envelope variance reveals underlying component degradation.

In [ ]:
loader.fit_regime_standardization(train_df)
train_norm = loader.standardize_by_regime(train_df)
test_norm = loader.standardize_by_regime(test_df)
train_feats = loader.extract_features(train_norm)
test_feats = loader.extract_features(test_norm)
print("Features extracted:", [c for c in train_feats.columns if "norm" in c][:6])

## 3. RUL Prediction & Conformal Uncertainty Bounds
Evaluating on holdout test engines with 90% finite-sample marginal coverage.

In [ ]:
rul_model = RULPredictor("../ml/models/rul_predictor.joblib")
conformal = ConformalPredictor()
conformal.load("../ml/models/conformal_predictor.joblib")
print("Conformal quantile q:", conformal.q_value)
